# Google Trends Related Pipeline — Step by Step

Monthly related **topics** and **queries** → analysis-ready monthly tables,
even when the collector extract is incomplete.

Mirrors `src/pipeline/google_trends/related_steps.py` and
`related_transform.py`. Runs on fixtures under
`tests/fixtures/google_trends/`.

**Pipeline:** scan → rename (`date`→`month`, `location`→`state_abbrev`,
`topic_title`→`related_title`) → coerce value → concat sources → optional
drop BR / empty → dedupe by source precedence → order

**Design choices (literature / project):**

| Choice | Why |
|---|---|
| Keep **monthly** grain | Related APIs are monthly; not a weekly surveillance panel |
| Same `state_abbrev` as search/SINAN/climate | One geography vocabulary |
| Partial extracts OK | ~18 h collector; 429 stops mid-run |
| Drop blank `related_title` by default | Placeholder rows when GT returns nothing |
| No expansion to epi-week | Modelling / vocabulary use; join on month if needed |
| **Concat** frozen seed + our extract | The seed is the published reference; our extract only extends the months it does not reach |
| Every row carries `source`; seed wins on overlap | One row per key, and the table agrees with the published dataset wherever that dataset reaches |

The two inputs were produced by **different clients** (R `gtrendsAPI` for the
seed, `pytrends` for ours), so they disagree by a point or two on the keys they
share. That is why precedence is explicit rather than "last write wins".


## Setup

In [1]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.google_trends.related_steps import (
    RelatedSource,
    build_queries,
    build_topics,
    drop_empty_related,
    rename_related_keys,
    scan_related_queries,
    scan_related_topics,
)
from pipeline.google_trends.spec import (
    MONTH,
    QUERY_OUTPUT_COLUMNS,
    RELATED_TITLE,
    SOURCE_EXTRACT,
    SOURCE_SEED,
    STATE_ABBREV,
    TOPIC_OUTPUT_COLUMNS,
)

pl.Config.set_tbl_rows(12)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "google_trends"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"
TOPICS = FIXTURES / "GoogleTrends_related_topic_sample.csv"
QUERIES = FIXTURES / "GoogleTrends_related_query_sample.csv"

# The steps take a *list* of (path, provenance) sources. In the real run that is
# our own extract plus the frozen Zenodo seed; the fixture shows a single source.
TOPIC_SOURCES = [RelatedSource(TOPICS, SOURCE_EXTRACT)]
QUERY_SOURCES = [RelatedSource(QUERIES, SOURCE_EXTRACT)]
TOPICS.exists(), QUERIES.exists()


(True, True)

## Step 1 — Scan related topics (raw extractor schema)


In [2]:
raw_topics = scan_related_topics(TOPICS).collect()
raw_topics


date,location,topic_title,topic_id,value,disease,key_symptom
str,str,str,str,str,str,bool
"""2020-01""","""AC""","""Dengue fever""","""/m/09wsg""","""100""","""Dengue""",false
"""2020-01""","""AC""","""Symptom""","""/m/0c1tj""","""40""","""Dengue""",true
"""2020-01""","""SP""",null,"""/m/09wsg""",null,"""Dengue""",false
"""2020-02""","""BR""","""Aedes aegypti""","""/m/012c4b""","""55""","""Dengue""",false


## Step 2 — Rename keys + drop empty related titles

Blank `topic_title` rows are placeholders (GT returned nothing for that
month/geo). Default transform drops them.


In [3]:
renamed = rename_related_keys(raw_topics.lazy())
kept = drop_empty_related(renamed, drop=True).collect()
{
    "raw_rows": raw_topics.height,
    "after_drop_empty": kept.height,
    "columns": kept.columns,
}


{'raw_rows': 4,
 'after_drop_empty': 3,
 'columns': ['month',
  'state_abbrev',
  'related_title',
  'topic_id',
  'value',
  'disease',
  'key_symptom']}

## Step 3 — `build_topics` / `build_queries` end-to-end


In [4]:
topics = build_topics(TOPIC_SOURCES, keep_br=True, drop_empty=True).collect()
queries = build_queries(QUERY_SOURCES, keep_br=True, drop_empty=True).collect()
assert tuple(topics.columns) == TOPIC_OUTPUT_COLUMNS
assert tuple(queries.columns) == QUERY_OUTPUT_COLUMNS
topics, queries


(shape: (3, 8)
 ┌─────────┬──────────────┬─────────┬───────────────┬───────────┬───────┬─────────────┬──────────┐
 │ month   ┆ state_abbrev ┆ disease ┆ related_title ┆ topic_id  ┆ value ┆ key_symptom ┆ source   │
 │ ---     ┆ ---          ┆ ---     ┆ ---           ┆ ---       ┆ ---   ┆ ---         ┆ ---      │
 │ str     ┆ str          ┆ str     ┆ str           ┆ str       ┆ f64   ┆ bool        ┆ str      │
 ╞═════════╪══════════════╪═════════╪═══════════════╪═══════════╪═══════╪═════════════╪══════════╡
 │ 2020-01 ┆ AC           ┆ Dengue  ┆ Dengue fever  ┆ /m/09wsg  ┆ 100.0 ┆ false       ┆ pytrends │
 │ 2020-01 ┆ AC           ┆ Dengue  ┆ Symptom       ┆ /m/0c1tj  ┆ 40.0  ┆ true        ┆ pytrends │
 │ 2020-02 ┆ BR           ┆ Dengue  ┆ Aedes aegypti ┆ /m/012c4b ┆ 55.0  ┆ false       ┆ pytrends │
 └─────────┴──────────────┴─────────┴───────────────┴───────────┴───────┴─────────────┴──────────┘,
 shape: (4, 6)
 ┌─────────┬──────────────┬─────────────┬────────────────────┬───────┬────────

## Integrity (partial coverage allowed)

`require_complete_ufs=False` by default — incomplete collectors must not fail
the transform.


In [5]:
from pipeline.google_trends.integrity import run_all_related

print("=== topics ===")
for check in run_all_related(topics, FU_PATH, kind="topics"):
    print(check)
print("=== queries ===")
for check in run_all_related(queries, FU_PATH, kind="queries"):
    print(check)


=== topics ===
[PASS] schema: 8 columns in published order (topics)
[PASS] unique_keys: 3 rows, all keys distinct
[PASS] month_format: all months match YYYY-MM
[PASS] states: 2 known locations (incomplete extract; missing 26 UFs)
[PASS] value_domain: value in [0, 100] or null
[PASS] related_titles: 3 non-empty related titles
[PASS] sources: pytrends=3
=== queries ===
[PASS] schema: 6 columns in published order (queries)
[PASS] unique_keys: 4 rows, all keys distinct
[PASS] month_format: all months match YYYY-MM
[PASS] states: 3 known locations (incomplete extract; missing 25 UFs)
[PASS] value_domain: value in [0, 100] or null
[PASS] related_titles: 4 non-empty related titles
[PASS] sources: pytrends=4


## Optional — peek at local extract + transform output


In [6]:
raw_dir = PROJECT_ROOT / "data" / "google_trends"
seed_dir = PROJECT_ROOT / "data" / "external" / "zenodo_17102575"
for path in (
    raw_dir / "GoogleTrends_related_topic.csv",
    raw_dir / "GoogleTrends_related_query.csv",
    seed_dir / "GoogleTrends_related_topic.csv",
    seed_dir / "GoogleTrends_related_query.csv",
    raw_dir / "GoogleTrends_related_topic_monthly.parquet",
    raw_dir / "GoogleTrends_related_query_monthly.parquet",
):
    if not path.exists():
        print(f"missing: {path.relative_to(PROJECT_ROOT)}")
        continue
    if path.suffix == ".parquet":
        df = pl.read_parquet(path)
    else:
        df = pl.read_csv(path, ignore_errors=True)
    print(f"{path.relative_to(PROJECT_ROOT)}: {df.height:,} rows, cols={df.columns}")


data/google_trends/GoogleTrends_related_topic.csv: 1,440 rows, cols=['topic_title', 'topic_id', 'date', 'location', 'value', 'disease', 'key_symptom']
data/google_trends/GoogleTrends_related_query.csv: 12,382 rows, cols=['topic_title', 'date', 'location', 'value', 'disease']
data/external/zenodo_17102575/GoogleTrends_related_topic.csv: 120,232 rows, cols=['topic_title', 'topic_id', 'date', 'location', 'value', 'disease', 'key_symptom']
data/external/zenodo_17102575/GoogleTrends_related_query.csv: 79,255 rows, cols=['topic_title', 'date', 'location', 'value', 'disease']
data/google_trends/GoogleTrends_related_topic_monthly.parquet: 119,964 rows, cols=['month', 'state_abbrev', 'disease', 'related_title', 'topic_id', 'value', 'key_symptom', 'source']
data/google_trends/GoogleTrends_related_query_monthly.parquet: 82,263 rows, cols=['month', 'state_abbrev', 'disease', 'related_title', 'value', 'source']


## Running it for real

```bash
# continue / resume collection (long)
uv run arboili gt-related

# reshape whatever is on disk today — concatenates the frozen seed + our extract
uv run arboili transform gt-related

# our extract only, ignoring the frozen seed
uv run arboili transform gt-related --no-seed
```

Methods: `src/pipeline/google_trends/README.md`. Seed provenance and caveats:
`data/external/zenodo_17102575/README.md`.
